In [1]:
from google.cloud import bigquery
import pandas as pd

# Construct a BigQuery client object.
client = bigquery.Client()

query = """
    SELECT *
    FROM `proj-sales-recommender-dev.sales_recommender_dev.project_relevance`
    WHERE confidence IS NOT NULL
"""

# Execute the query and load the results into a pandas DataFrame.
df = client.query(query).to_dataframe()

# Display the first few rows of the DataFrame.
df.head()

,project_id,relevance,reasoning,search_id,territory_id,time_created,source,modified_on,distance,confidence
0,9202892,High,This is a new construction project for a K-12 ...,0b3ab0d4-ec94-4ad2-974e-997e3055b839,None,2025-04-23 06:41:35,construct_connect,2025-04-29 21:33:13.452993+00:00,NaN,0.90
1,9202892,High,This is a new construction project for a K-12 ...,70e6309f-5d11-f011-9988-000d3a5a18fb,None,2025-04-23 06:41:35,construct_connect,2025-04-29 21:33:16.920857+00:00,NaN,0.90
2,9202892,High,This is a new construction project for a K-12 ...,16700387-5d11-f011-9988-000d3a5a18fb,None,2025-04-23 06:41:35,construct_connect,2025-04-29 21:33:16.386679+00:00,NaN,0.90
3,9202892,High,This is a new construction project for a K-12 ...,dd84e5a5-5d11-f011-9988-000d3a5a18fb,None,2025-04-23 06:41:35,construct_connect,2025-04-29 21:33:17.105623+00:00,NaN,0.85
4,1004561789,Moderate,The project is a renovation of an educational ...,70e6309f-5d11-f011-9988-000d3a5a18fb,717aa518-7898-e911-a833-000d3a315225,2025-04-23 06:41:35,construct_connect,2025-04-29 21:36:22.341843+00:00,1.63,0.80


In [7]:
df['relevance'].value_counts()

relevance
High            5033
Moderate        2337
Low              845
Very High        541
Not Relevant       8
Name: count, dtype: int64

In [66]:
import pandas as pd

# --- Sampling Function ---
# Group by 'relevance', drop duplicate 'project_id' within each group,
# then take a sample of 10 from each group.
def sample_unique_projects(group):
    # Sample 10 or fewer if the group size is less than 10
    sample_size = min(10, len(group))
    # Using random_state for reproducibility
    return group.sample(n=sample_size, random_state=42)

# --- Low Confidence Sample (confidence < 0.8) ---
print("--- Processing Low Confidence Sample (confidence < 0.8) ---")
# Filter for confidence < 0.8
df_low_confidence = df[df['confidence'] < 0.9].copy()
# Shuffle the DataFrame first
random_sample_df = df_low_confidence.sample(frac=1, random_state=42)

# Display the number of rows with confidence < 0.8
print(f"Number of rows with confidence < 0.9: {len(random_sample_df)}")
# Display value counts for 'relevance' after filtering
print("Value counts for 'relevance' (confidence < 0.9:")
print(random_sample_df['relevance'].value_counts())
# Display number of rows dropped (i.e., those with confidence >= 0.8)
print(f"Number of rows with confidence >= 0.9: {len(df) - len(random_sample_df)}")

# Drop duplicates based on 'project_id' to ensure unique projects
unique_low_confidence_df = random_sample_df.drop_duplicates(subset=['project_id'])
print(f"\nNumber of unique projects with confidence < 0.9: {len(unique_low_confidence_df)}")

# Apply the sampling function to each relevance group
random_sample_df = unique_low_confidence_df.groupby('relevance', group_keys=False).apply(sample_unique_projects)
random_sample_df = random_sample_df.reset_index(drop=True)

# Display the resulting low confidence sample DataFrame information and head
print("\nLow Confidence Sample DataFrame Info:")
random_sample_df.info()
print("\nLow Confidence Sample DataFrame Head:")
print(random_sample_df.head())
print("\nValue counts for 'relevance' in the low confidence sample:")
print(random_sample_df['relevance'].value_counts())


# --- High Confidence Sample (confidence >= 0.8) ---
print("\n--- Processing High Confidence Sample (confidence >= 0.9) ---")
# Filter for confidence >= 0.8
df_high_confidence = df[df['confidence'] >= 0.9].copy()
# Shuffle the DataFrame first
high_confidence_shuffled_df = df_high_confidence.sample(frac=1, random_state=42)

# Display the number of rows with confidence >= 0.8
print(f"Number of rows with confidence >= 0.9: {len(high_confidence_shuffled_df)}")
# Display value counts for 'relevance' after filtering
print("Value counts for 'relevance' (confidence >= 0.9):")
print(high_confidence_shuffled_df['relevance'].value_counts())

# Drop duplicates based on 'project_id' to ensure unique projects
unique_high_confidence_df = high_confidence_shuffled_df.drop_duplicates(subset=['project_id'])
print(f"\nNumber of unique projects with confidence >= 0.9: {len(unique_high_confidence_df)}")

# Apply the sampling function to each relevance group
high_confidence_sample_df = unique_high_confidence_df.groupby('relevance', group_keys=False).apply(sample_unique_projects)
high_confidence_sample_df = high_confidence_sample_df.reset_index(drop=True)

# Display the resulting high confidence sample DataFrame information and head
print("\nHigh Confidence Sample DataFrame Info:")
high_confidence_sample_df.info()
print("\nHigh Confidence Sample DataFrame Head:")
print(high_confidence_sample_df.head())
print("\nValue counts for 'relevance' in the high confidence sample:")
print(high_confidence_sample_df['relevance'].value_counts())

# Note: The variable 'random_sample_df' now holds the low-confidence sample.
# The variable 'high_confidence_sample_df' holds the high-confidence sample.
# The subsequent cells (like cell 4) use 'random_sample_df'.
# If you intend to use the high-confidence sample later, you might need to
# rename 'random_sample_df' in those cells or assign high_confidence_sample_df
# to random_sample_df before running them, depending on your goal.

--- Processing Low Confidence Sample (confidence < 0.8) ---
Number of rows with confidence < 0.9: 6031
Value counts for 'relevance' (confidence < 0.9:
relevance
High            2877
Moderate        2336
Low              792
Very High         22
Not Relevant       4
Name: count, dtype: int64
Number of rows with confidence >= 0.9: 2733

Number of unique projects with confidence < 0.9: 784

Low Confidence Sample DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39 entries, 0 to 38
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype              
---  ------        --------------  -----              
 0   project_id    39 non-null     Int64              
 1   relevance     39 non-null     object             
 2   reasoning     39 non-null     object             
 3   search_id     39 non-null     object             
 4   territory_id  27 non-null     object             
 5   time_created  39 non-null     datetime64[us]     
 6   source        39 non-nu

/tmp/ipykernel_5098/2340737390.py:32: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  random_sample_df = unique_low_confidence_df.groupby('relevance', group_keys=False).apply(sample_unique_projects)
/tmp/ipykernel_5098/2340737390.py:62: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  high_confidence_sample_df = unique_high_confidence_df.groupby('relevance', group_keys=False).apply(sample_unique_projects)


In [80]:
2733/(2733+6031)

0.3118439068918302

In [67]:
random_sample_df = pd.concat([random_sample_df, high_confidence_sample_df])
random_sample_df

,project_id,relevance,reasoning,search_id,territory_id,time_created,source,modified_on,distance,confidence
0,1007508545,High,The project is for a new educational facility ...,0b3ab0d4-ec94-4ad2-974e-997e3055b839,ce901cce-3e71-eb11-a812-000d3a3743d1,2025-04-23 06:41:35,construct_connect,2025-04-29 21:28:30.964169+00:00,39.48,0.85
1,1007552539,High,This project is classified as high relevance b...,dd84e5a5-5d11-f011-9988-000d3a5a18fb,7679a998-95ed-ed11-8849-00224808d95d,2025-04-23 06:41:35,construct_connect,2025-04-29 21:34:43.959867+00:00,6.80,0.85
2,1005258112,High,This project is classified as high relevance b...,dd84e5a5-5d11-f011-9988-000d3a5a18fb,c9df49dc-04b6-ee11-a569-00224808d025,2025-04-23 06:41:35,construct_connect,2025-04-29 21:35:02.979865+00:00,17.26,0.85
3,1007499510,High,This is a renovation project at Amos Hospital ...,16700387-5d11-f011-9988-000d3a5a18fb,None,2025-04-30 06:42:22,construct_connect,2025-04-30 07:44:04.273732+00:00,NaN,0.80
4,1007571730,High,This project is classified as high relevance d...,16700387-5d11-f011-9988-000d3a5a18fb,457aa518-7898-e911-a833-000d3a315225,2025-04-23 06:41:35,construct_connect,2025-04-29 21:27:59.984132+00:00,18.80,0.85
...,...,...,...,...,...,...,...,...,...,...
28,1007559902,Very High,This is a new construction project for a deten...,70e6309f-5d11-f011-9988-000d3a5a18fb,None,2025-04-30 06:42:22,construct_connect,2025-04-30 07:13:55.128500+00:00,NaN,0.90
29,1007603707,Very High,"This is a new construction project for a 51,60...",16700387-5d11-f011-9988-000d3a5a18fb,91bfe365-92ed-ed11-8849-00224808db35,2025-04-30 06:42:22,construct_connect,2025-04-30 07:35:32.978865+00:00,1.28,0.90
30,1007603122,Very High,"This project is classified as ""Very High"" rele...",70e6309f-5d11-f011-9988-000d3a5a18fb,d779a518-7898-e911-a833-000d3a315225,2025-04-30 06:42:22,construct_connect,2025-04-30 07:20:56.633522+00:00,28.29,0.90
31,1007453449,Very High,This project involves the new construction and...,70e6309f-5d11-f011-9988-000d3a5a18fb,bf9b8bb4-59ac-ec11-9840-0022480c5418,2025-04-30 06:42:22,construct_connect,2025-04-30 07:44:49.421112+00:00,7.54,0.90


In [75]:
import os
from google.cloud import bigquery
import pandas as pd

# Assuming 'client' is your authenticated BigQuery client from the first cell
# Assuming 'random_sample_df' contains your sampled project relevance data

# --- Configuration (Adapt these from your settings if needed) ---
# You might need to define these based on your environment or replace with actual values
# Example: settings.BIGQUERY_DATASET = "proj-sales-recommender-dev.sales_recommender_dev"
# Example: settings.CC_FEED_TABLE_ID = "construct_connect_feed" # Replace with your actual table name

# Replace with your actual dataset and table names
BIGQUERY_DATASET = "proj-sales-recommender-dev.sales_recommender_dev" # Replace if different
CC_FEED_TABLE_ID = "construct_connect_feed" # Replace with your actual CC table name
CC_TABLE_FULL_PATH = f"`{BIGQUERY_DATASET}.{CC_FEED_TABLE_ID}`"

# Get the list of unique project IDs from your sample
project_ids_list = random_sample_df['project_id'].unique().tolist()

# Convert list of integers to a comma-separated string for the SQL IN clause
project_ids_str = ', '.join(map(str, project_ids_list))

# --- Build the Query (Adapted from get_cc_row) ---
# Note: This query fetches data only for the project IDs in your sample.
# It doesn't filter by sourcefile or timeCreated like the original function.
cc_query = f"""
WITH exploded AS (
    SELECT
        ProjectID,
        company.CompanyID,
        company.Name AS company_name,
        company.BiddingRole,
        company,
        contact,
        sourceFileCreationTime,
    FROM {CC_TABLE_FULL_PATH},
    UNNEST(Companies) AS company_wrap,
    UNNEST(company_wrap.Company) AS company,
    UNNEST(company.contacts.contact) AS contact
    WHERE ProjectID IN ({project_ids_str}) -- Filter by project IDs in the sample
),
gc_contact AS (
    SELECT
        ProjectID,
        LEFT(company.name, 50) AS companyname,
        LEFT(company.Addresses.Address[SAFE_OFFSET(0)].City, 50) AS address1_city,
        LEFT(company.Addresses.Address[SAFE_OFFSET(0)].StateProvince, 50) AS address1_stateorprovince,
        LEFT(company.Addresses.Address[SAFE_OFFSET(0)].County, 50) AS address1_county,
        LEFT(company.Email, 50) AS emailaddress1,
        (
            SELECT ph.PhoneNumnber
            FROM UNNEST(company.Phones.phone) AS ph
            WHERE ph.PhoneType = 'Company Phone Number'
            LIMIT 1 -- Ensure only one phone number is selected if multiple exist
        ) AS mobilephone,
        LEFT(SPLIT(contact.Name, ' ')[SAFE_OFFSET(0)], 50) AS firstname,
        LEFT(SPLIT(contact.Name, ' ')[SAFE_OFFSET(1)], 50) AS lastname
    FROM exploded
    WHERE BiddingRole = "General Contractor"
    QUALIFY ROW_NUMBER() OVER(PARTITION BY ProjectID ORDER BY contact.Name) = 1 -- Added ORDER BY for deterministic selection
),
contact_jsons AS (
    SELECT
        ProjectID,
        CompanyID,
        -- Simplified JSON creation for broader compatibility, adjust if specific format needed
        TO_JSON_STRING(STRUCT(
            company_name AS `Company Name`,
            contact.Name AS `Name`,
            contact.Email AS `Email`,
            contact.PhoneNumber AS `PhoneNumber`
        )) AS contact_json
    FROM
        exploded
    -- Removed the BiddingRole filter here to potentially include all bidders if needed
),
contact_group AS (
    SELECT
        ProjectID,
        CompanyID,
        STRING_AGG(contact_json, ', ') AS contact -- Changed separator for clarity
    FROM
        contact_jsons
    GROUP BY
        ProjectID,
        CompanyID
),
bidder_group AS (
    SELECT
        ProjectID,
        STRING_AGG(contact, '\\n') AS bidder_lead_list -- Using STRING_AGG on the aggregated contacts per company
    FROM
        contact_group
    GROUP BY
        ProjectID
)
SELECT
    cc.ProjectID AS project_id, -- Alias to match the column name in random_sample_df
    cc.Title AS arb_project_name,
    cc.Stage AS fbm_projectstage,
    cc.Parameters_Parameter_BidDate AS fbm_biddate,
    cc.Valuation_Value AS estimatedamount,
    cc.Parameters_Parameter_CommenceDate AS fbm_startdate,
    cc.sourceFileCreationTime AS fbm_lastsyncdate,
    cc.Addresses_Address[SAFE_OFFSET(0)].AddressLine1 AS arb_project_line1,
    cc.Addresses_Address[SAFE_OFFSET(0)].AddressLine2 AS arb_project_line2,
    cc.Addresses_Address[SAFE_OFFSET(0)].City AS arb_project_city,
    cc.Addresses_Address[SAFE_OFFSET(0)].StateProvince AS arb_project_stateorprovince,
    cc.Addresses_Address[SAFE_OFFSET(0)].ZipPostalCode AS arb_project_postalcode,
    cc.Addresses_Address[SAFE_OFFSET(0)].CountryRegion AS arb_project_country,
    SUBSTR(ARRAY_TO_STRING(cc.Details_Detail_Scope, '\\n'), 1, 2000) AS fbm_projectdescription, -- Apply substring limit
    (SELECT STRING_AGG(pc.Name, ', ') FROM UNNEST(cc.ParentCategories_ParentCategory) pc) AS fbm_projectcategories,
    cc.URL AS fbm_externalleadurl,
    cc.DocumentAvailability_Plans AS fbm_plansavailable,
    cc.DocumentAvailability_Specs AS fbm_specificationsavailable,
    cc.DocumentAvailability_Addenda AS fbm_addendaavailable,
    cc.Parameters_Parameter_WorkType AS fbm_worktype,
    CAST(cc.Parameters_Parameter_FloorArea AS STRING) AS arb_totalsquarefootage,
    gc_contact.* EXCEPT(ProjectID), -- Select all columns from gc_contact except ProjectID
    bidder_lead_list AS fbm_externalleadbidders
FROM
    {CC_TABLE_FULL_PATH} cc
LEFT JOIN bidder_group
    ON cc.ProjectID = bidder_group.ProjectID
LEFT JOIN gc_contact
    ON cc.ProjectID = gc_contact.ProjectID
WHERE
    cc.ProjectID IN ({project_ids_str}); -- Filter the main table by project IDs in the sample
"""

# --- Execute Query and Load Data ---
print("Fetching ConstructConnect data for sampled projects...")
# Ensure client is initialized (should be from your first cell)
if 'client' not in locals():
    client = bigquery.Client()

cc_df = client.query(cc_query).to_dataframe()
cc_df = cc_df.sort_values(by=['project_id', 'fbm_lastsyncdate'], ascending=[True, False]).drop_duplicates(subset=['project_id'], keep='first')

# --- Data Type Conversion (similar to get_cc_row) ---
if 'estimatedamount' in cc_df.columns:
        # Convert to numeric, coercing errors to NaN, then fill NaN with None (or 0 if preferred)
    cc_df['estimatedamount'] = pd.to_numeric(cc_df['estimatedamount'], errors='coerce')
    # cc_df['estimatedamount'] = cc_df['estimatedamount'].fillna(0) # Option to fill with 0

print(f"Fetched {len(cc_df)} rows from ConstructConnect table.")

# --- Merge DataFrames ---
print("Merging sampled relevance data with ConstructConnect data...")
# Use a left merge to keep all rows from random_sample_df and add matching cc data
merged_df = pd.merge(random_sample_df, cc_df, on='project_id', how='left')

print("Merge complete.")
print("\nMerged DataFrame Info:")
merged_df.info()
print("\nMerged DataFrame Head:")
print(merged_df.head())


# Display the first few rows of the merged DataFrame
merged_df.head()

Fetching ConstructConnect data for sampled projects...
Fetched 69 rows from ConstructConnect table.
Merging sampled relevance data with ConstructConnect data...
Merge complete.

Merged DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 72 entries, 0 to 71
Data columns (total 39 columns):
 #   Column                       Non-Null Count  Dtype              
---  ------                       --------------  -----              
 0   project_id                   72 non-null     Int64              
 1   relevance                    72 non-null     object             
 2   reasoning                    72 non-null     object             
 3   search_id                    72 non-null     object             
 4   territory_id                 53 non-null     object             
 5   time_created                 72 non-null     datetime64[us]     
 6   source                       72 non-null     object             
 7   modified_on                  72 non-null     datetime64[us, U

,project_id,relevance,reasoning,search_id,territory_id,time_created,source,modified_on,distance,confidence,arb_project_name,fbm_projectstage,fbm_biddate,estimatedamount,fbm_startdate,fbm_lastsyncdate,arb_project_line1,arb_project_line2,arb_project_city,arb_project_stateorprovince,arb_project_postalcode,arb_project_country,fbm_projectdescription,fbm_projectcategories,fbm_externalleadurl,fbm_plansavailable,fbm_specificationsavailable,fbm_addendaavailable,fbm_worktype,arb_totalsquarefootage,companyname,address1_city,address1_stateorprovince,address1_county,emailaddress1,mobilephone,firstname,lastname,fbm_externalleadbidders
0,1007508545,High,The project is for a new educational facility ...,0b3ab0d4-ec94-4ad2-974e-997e3055b839,ce901cce-3e71-eb11-a812-000d3a3743d1,2025-04-23 06:41:35,construct_connect,2025-04-29 21:28:30.964169+00:00,39.48,0.85,San Marcos ISD Natatorium,Post Bid,2025-03-13,17478750.0,2025-05-19,2025-04-23T06:41:35.174Z,2601 Rattler Rd,None,San Marcos,TX,78666,UNITED STATES,Site work and new construction of an education...,EDUCATIONAL,http://insight.cmdgroup.com/SingleSignOn/Proje...,False,False,False,New,25500,None,None,None,None,None,None,None,None,"{""Company Name"":""Stantec - San Antonio"",""Name""..."
1,1007552539,High,This project is classified as high relevance b...,dd84e5a5-5d11-f011-9988-000d3a5a18fb,7679a998-95ed-ed11-8849-00224808d95d,2025-04-23 06:41:35,construct_connect,2025-04-29 21:34:43.959867+00:00,6.80,0.85,2023 Capital Improvement Project Phase 2B - Pl...,Post Bid,2025-04-22,1300000.0,2025-05-13,2025-04-23T06:41:35.174Z,Multiple Locations,None,Baldwinsville,NY,13027,UNITED STATES,Renovation of a mixed-use development in Baldw...,EDUCATIONAL,http://insight.cmdgroup.com/SingleSignOn/Proje...,True,True,True,Alteration,None,None,None,None,None,None,None,None,None,"{""Company Name"":""Plan & Print Systems, Inc."",""..."
2,1005258112,High,This project is classified as high relevance b...,dd84e5a5-5d11-f011-9988-000d3a5a18fb,c9df49dc-04b6-ee11-a569-00224808d025,2025-04-23 06:41:35,construct_connect,2025-04-29 21:35:02.979865+00:00,17.26,0.85,"MS/HS 141 (X) Exterior Masonry, Parapets, Roofs",General Contractor Award,2019-06-20,7200000.0,2019-08-19,2025-04-23T06:41:35.174Z,660 W 237th St,None,Bronx,NY,10463,UNITED STATES,Renovation of a mixed-use development in Bronx...,EDUCATIONAL,http://insight.cmdgroup.com/SingleSignOn/Proje...,True,True,False,Alteration,None,Chapman & Evans,Long Island City,NY,Queens,chapmanandevans@yahoo.com,718 472-7433,Aleksandr,Trofimov,"{""Company Name"":""Franco Belli Plumbing & Heati..."
3,1007499510,High,This is a renovation project at Amos Hospital ...,16700387-5d11-f011-9988-000d3a5a18fb,None,2025-04-30 06:42:22,construct_connect,2025-04-30 07:44:04.273732+00:00,NaN,0.80,Redevelopment - Sterile Preparation Areas - Am...,Biddate Set,2025-05-29,1200000.0,2025-06-30,2025-04-30T06:42:22.629Z,622 4e Rue O,None,Amos,QC,J9T 2S2,CANADA,"Renovation of a medical facility in Amos, Queb...",MEDICAL,http://insight.cmdgroup.com/SingleSignOn/Proje...,False,True,False,Alteration,None,Hardy Construction (2645-3530 QUEBEC INC),Amos,QC,"Abitibi, comté",info@hardy-construction.ca,819 732-1493,Achat,Achat,"{""Company Name"":""Construction Filiatrault Inc""..."
4,1007571730,High,This project is classified as high relevance d...,16700387-5d11-f011-9988-000d3a5a18fb,457aa518-7898-e911-a833-000d3a315225,2025-04-23 06:41:35,construct_connect,2025-04-29 21:27:59.984132+00:00,18.80,0.85,RFQ Construction Manager at Risk - Secure Entr...,Biddate Set,2025-06-04,17180000.0,2026-02-04,2025-04-23T06:41:35.174Z,Multiple Locations,None,Center Point,IA,52213,UNITED STATES,"Demolition, site work, renovation and addition...","CIVIL, COMMUNITY, EDUCATIONAL",http://insight.cmdgroup.com/SingleSignOn/Proje...,True,True,False,Addition/Alteration,None,None,None,None,None,None,None,None,None,"{""Company Name"":""Center Point-Urbana Community..."


In [76]:
query = """
    SELECT *
    FROM `proj-sales-recommender-dev.sales_recommender_dev.searches`
"""

# Execute the query and load the results into a pandas DataFrame.
searches = client.query(query).to_dataframe()

# Display the first few rows of the DataFrame.
searches.head()

,id,name,category,boolean,upsert_time
0,0b3ab0d4-ec94-4ad2-974e-997e3055b839,CEILINGS,866070000,(armstrong NEAR ceiling*) OR rockfon OR (usg N...,2025-05-01 16:34:17.893623
1,5ee2ab20-266c-4fc2-bac9-f784353df5a6,FRP,866070002,acrovyn OR (frp NEAR wall) OR (frl NEAR wall) ...,2025-05-01 16:34:17.893623
2,56ff8186-f036-4963-a398-92d113ea2408,EIFS,None,eifs NEAR adex) OR (eifs NEAR dryvit) OR (eifs...,2025-05-01 16:34:17.893623
3,70e6309f-5d11-f011-9988-000d3a5a18fb,Core,None,(usg NEAR drywall) OR (usg NEAR gypsum) OR ('u...,2025-05-01 16:34:17.893623
4,16700387-5d11-f011-9988-000d3a5a18fb,Drywall,None,(usg NEAR drywall) OR (usg NEAR gypsum) OR ('u...,2025-05-01 16:34:17.893623


In [77]:
pd.options.display.max_columns = None  # Show all columns in the DataFrame

merged_df = merged_df.merge(searches[['id', 'name']],  left_on='search_id', right_on='id')
merged_df

,project_id,relevance,reasoning,search_id,territory_id,time_created,source,modified_on,distance,confidence,arb_project_name,fbm_projectstage,fbm_biddate,estimatedamount,fbm_startdate,fbm_lastsyncdate,arb_project_line1,arb_project_line2,arb_project_city,arb_project_stateorprovince,arb_project_postalcode,arb_project_country,fbm_projectdescription,fbm_projectcategories,fbm_externalleadurl,fbm_plansavailable,fbm_specificationsavailable,fbm_addendaavailable,fbm_worktype,arb_totalsquarefootage,companyname,address1_city,address1_stateorprovince,address1_county,emailaddress1,mobilephone,firstname,lastname,fbm_externalleadbidders,id,name
0,1007508545,High,The project is for a new educational facility ...,0b3ab0d4-ec94-4ad2-974e-997e3055b839,ce901cce-3e71-eb11-a812-000d3a3743d1,2025-04-23 06:41:35,construct_connect,2025-04-29 21:28:30.964169+00:00,39.48,0.85,San Marcos ISD Natatorium,Post Bid,2025-03-13,17478750.0,2025-05-19,2025-04-23T06:41:35.174Z,2601 Rattler Rd,None,San Marcos,TX,78666,UNITED STATES,Site work and new construction of an education...,EDUCATIONAL,http://insight.cmdgroup.com/SingleSignOn/Proje...,False,False,False,New,25500,None,None,None,None,None,None,None,None,"{""Company Name"":""Stantec - San Antonio"",""Name""...",0b3ab0d4-ec94-4ad2-974e-997e3055b839,CEILINGS
1,1007552539,High,This project is classified as high relevance b...,dd84e5a5-5d11-f011-9988-000d3a5a18fb,7679a998-95ed-ed11-8849-00224808d95d,2025-04-23 06:41:35,construct_connect,2025-04-29 21:34:43.959867+00:00,6.80,0.85,2023 Capital Improvement Project Phase 2B - Pl...,Post Bid,2025-04-22,1300000.0,2025-05-13,2025-04-23T06:41:35.174Z,Multiple Locations,None,Baldwinsville,NY,13027,UNITED STATES,Renovation of a mixed-use development in Baldw...,EDUCATIONAL,http://insight.cmdgroup.com/SingleSignOn/Proje...,True,True,True,Alteration,None,None,None,None,None,None,None,None,None,"{""Company Name"":""Plan & Print Systems, Inc."",""...",dd84e5a5-5d11-f011-9988-000d3a5a18fb,Steel Sales
2,1005258112,High,This project is classified as high relevance b...,dd84e5a5-5d11-f011-9988-000d3a5a18fb,c9df49dc-04b6-ee11-a569-00224808d025,2025-04-23 06:41:35,construct_connect,2025-04-29 21:35:02.979865+00:00,17.26,0.85,"MS/HS 141 (X) Exterior Masonry, Parapets, Roofs",General Contractor Award,2019-06-20,7200000.0,2019-08-19,2025-04-23T06:41:35.174Z,660 W 237th St,None,Bronx,NY,10463,UNITED STATES,Renovation of a mixed-use development in Bronx...,EDUCATIONAL,http://insight.cmdgroup.com/SingleSignOn/Proje...,True,True,False,Alteration,None,Chapman & Evans,Long Island City,NY,Queens,chapmanandevans@yahoo.com,718 472-7433,Aleksandr,Trofimov,"{""Company Name"":""Franco Belli Plumbing & Heati...",dd84e5a5-5d11-f011-9988-000d3a5a18fb,Steel Sales
3,1007499510,High,This is a renovation project at Amos Hospital ...,16700387-5d11-f011-9988-000d3a5a18fb,None,2025-04-30 06:42:22,construct_connect,2025-04-30 07:44:04.273732+00:00,NaN,0.80,Redevelopment - Sterile Preparation Areas - Am...,Biddate Set,2025-05-29,1200000.0,2025-06-30,2025-04-30T06:42:22.629Z,622 4e Rue O,None,Amos,QC,J9T 2S2,CANADA,"Renovation of a medical facility in Amos, Queb...",MEDICAL,http://insight.cmdgroup.com/SingleSignOn/Proje...,False,True,False,Alteration,None,Hardy Construction (2645-3530 QUEBEC INC),Amos,QC,"Abitibi, comté",info@hardy-construction.ca,819 732-1493,Achat,Achat,"{""Company Name"":""Construction Filiatrault Inc""...",16700387-5d11-f011-9988-000d3a5a18fb,Drywall
4,1007571730,High,This project is classified as high relevance d...,16700387-5d11-f011-9988-000d3a5a18fb,457aa518-7898-e911-a833-000d3a315225,2025-04-23 06:41:35,construct_connect,2025-04-29 21:27:59.984132+00:00,18.80,0.85,RFQ Construction Manager at Risk - Secure Entr...,Biddate Set,2025-06-04,17180000.0,2026-02-04,2025-04-23T06:41:35.174Z,Multiple Locations,None,Center Point,IA,52213,UNITED STATES,"Demolition, site work, renovation and addition...","CIVIL, COMMUNITY, EDUCATIONAL",http://insight.cmdgroup.com/SingleSignOn/Proje...,True,True,Fal

In [78]:
final_df = merged_df[['project_id', 'name', 'relevance', 'reasoning', 
           'confidence', 'arb_project_name', 'fbm_projectdescription', 
           'fbm_projectcategories', 'fbm_externalleadurl', 
           'fbm_plansavailable', 'fbm_specificationsavailable', 'fbm_addendaavailable', 
           'fbm_worktype', 'arb_totalsquarefootage', 'fbm_externalleadbidders',
           'address1_city', 'address1_stateorprovince', 'address1_county', 'emailaddress1', 
           'mobilephone', 'firstname', 'lastname'
           ]]
final_df

,project_id,name,relevance,reasoning,confidence,arb_project_name,fbm_projectdescription,fbm_projectcategories,fbm_externalleadurl,fbm_plansavailable,fbm_specificationsavailable,fbm_addendaavailable,fbm_worktype,arb_totalsquarefootage,fbm_externalleadbidders,address1_city,address1_stateorprovince,address1_county,emailaddress1,mobilephone,firstname,lastname
0,1007508545,CEILINGS,High,The project is for a new educational facility ...,0.85,San Marcos ISD Natatorium,Site work and new construction of an education...,EDUCATIONAL,http://insight.cmdgroup.com/SingleSignOn/Proje...,False,False,False,New,25500,"{""Company Name"":""Stantec - San Antonio"",""Name""...",None,None,None,None,None,None,None
1,1007552539,Steel Sales,High,This project is classified as high relevance b...,0.85,2023 Capital Improvement Project Phase 2B - Pl...,Renovation of a mixed-use development in Baldw...,EDUCATIONAL,http://insight.cmdgroup.com/SingleSignOn/Proje...,True,True,True,Alteration,None,"{""Company Name"":""Plan & Print Systems, Inc."",""...",None,None,None,None,None,None,None
2,1005258112,Steel Sales,High,This project is classified as high relevance b...,0.85,"MS/HS 141 (X) Exterior Masonry, Parapets, Roofs",Renovation of a mixed-use development in Bronx...,EDUCATIONAL,http://insight.cmdgroup.com/SingleSignOn/Proje...,True,True,False,Alteration,None,"{""Company Name"":""Franco Belli Plumbing & Heati...",Long Island City,NY,Queens,chapmanandevans@yahoo.com,718 472-7433,Aleksandr,Trofimov
3,1007499510,Drywall,High,This is a renovation project at Amos Hospital ...,0.80,Redevelopment - Sterile Preparation Areas - Am...,"Renovation of a medical facility in Amos, Queb...",MEDICAL,http://insight.cmdgroup.com/SingleSignOn/Proje...,False,True,False,Alteration,None,"{""Company Name"":""Construction Filiatrault Inc""...",Amos,QC,"Abitibi, comté",info@hardy-construction.ca,819 732-1493,Achat,Achat
4,1007571730,Drywall,High,This project is classified as high relevance d...,0.85,RFQ Construction Manager at Risk - Secure Entr...,"Demolition, site work, renovation and addition...","CIVIL, COMMUNITY, EDUCATIONAL",http://insight.cmdgroup.com/SingleSignOn/Proje...,True,True,False,Addition/Alteration,None,"{""Company Name"":""Center Point-Urbana Community...",None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67,1007559902,Core,Very High,This is a new construction project for a deten...,0.90,Mono County New Jail New Construction,Site work and new construction of a detention ...,GOVERNMENT,http://insight.cmdgroup.com/SingleSignOn/Proje...,True,True,True,New,None,"{""Company Name"":""Lionakis - Sacramento"",""Name""...",Woodland,CA,Yolo,Estimating@BrowardBuilders.com,530 666-5635,Aaron,Houck
68,1007603707,Drywall,Very High,"This is a new construction project for a 51,60...",0.90,Construct New Computer Science Building,Site work and new construction of an education...,EDUCATIONAL,http://insight.cmdgroup.com/SingleSignOn/Proje...,True,True,False,New,51600,"{""Company Name"":""BET Consultants"",""Name"":""Asho...",New York,NY,New York,estimating@Vernon.net,212 421-2000,Adrian,A.
69,1007603122,Core,Very High,"This project is classified as ""Very High"" rele...",0.90,Jack C Hays HS 2025 Additions & Renovation,"Site work, renovation and addition to an educa...",EDUCATIONAL,http://insight.cmdgroup.com/SingleSignOn/Proje...,True,True,False,Addition/Alteration,54674,"{""Company Name"":""Huckabee & Associates Inc - F...",None,None,None,None,None,None,None
70,1007453449,Core,Very High,This project involves the new construction and...,0.90,RFQ D/B - Form an Alliance for North York Gene...,"Site work, new construction and renovation of ...",MEDICAL,http://insight.cmdgroup.com/SingleSignOn/Proje...,False,True,True,New,845670,"{""Company Name"":""Merx"",""Name"":""Buyer Support"",...",None,None,None,None,None,None,None


In [79]:
final_df[final_df['confidence'] < 0.9].sort_values(by='relevance').to_csv('low_confidence_sample.csv', index=False)
final_df[final_df['confidence'] >= 0.9].sort_values(by='relevance').to_csv('high_confidence_sample.csv', index=False)